# Import libs



In [1]:
import os

import numpy as np

import math

import pandas as pd

import matplotlib.pyplot as plt

from mpl_toolkits.mplot3d import Axes3D

import tensorflow as tf

In [2]:
cwd = os.getcwd()

# Denoising Filters



In [3]:
BUTTER_LOWPASS = "Butterworth Lowpass"
SAVITZKY_GOLAY = "Savitzky-Golay"
GAUSSIAN = "Gaussian"
WAVELET = "Wavelet"
WIENER = "Wiener"
CHEBYSHEV = "Chebyshev"
KALMAN = "Kalman"
BUTTER_CHEBYSHEV = "Butterworth Lowpass & Chebyshev"

In [4]:
%pip install PyWavelets
%pip install filterpy

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 1.5 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110458 sha256=d9fbb50ba381cf878f9b48352d308b2eb81d5e1913eb9b307cabaa859d666a6c
  Stored in directory: /root/.cache/pip/wheels/0f/0c/ea/218f266af4ad626897562199fbbcba521b8497303200186102
Successfully built filterpy
Note: you may need to restart the kernel to use updated packages.


In [5]:
from scipy.signal import butter, filtfilt, savgol_filter
from scipy.ndimage import gaussian_filter1d
from scipy.signal import medfilt, wiener, cheby1, filtfilt
import pywt
from filterpy.kalman import KalmanFilter
from filterpy.common import Q_discrete_white_noise

Butterworth Low-pass Filter



In [6]:
def apply_butter_lowpass_filter(df, column_name, cutoff=7, fs=100, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="low", analog=False)
    return filtfilt(b, a, df[column_name].values).astype("float32")

Savitzky-Golay Filter



In [7]:
def apply_savitzky_golay_filter(df, column_name, window_length=11, polyorder=2):
    return savgol_filter(df[column_name].values, window_length, polyorder).astype("float32")

Gaussian Filter



In [8]:
def apply_gaussian_filter(df, column_name, sigma=1.5):
    return gaussian_filter1d(df[column_name].values, sigma).astype("float32")

Wavelet Denoising



In [9]:
def apply_wavelet_denoising(df, column_name, wavelet, level):
    data = df[column_name].values

    coeffs = pywt.wavedec(data, wavelet, level=level)
    threshold = np.sqrt(2 * np.log(len(data)))
    denoised_coeffs = [pywt.threshold(c, value=threshold, mode='soft') for c in coeffs]
    denoised_data = pywt.waverec(denoised_coeffs, wavelet)
    return denoised_data[:len(data)].astype("float32")

Wiener Filter

In [10]:
def apply_wiener_filter(df, column_name):
    return wiener(df[column_name].values).astype("float32")

Chebyshev Filter

In [11]:
def apply_chebyshev_filter(df, column_name, cutoff=7, fs=100, order=3, rp=0.5):
  nyq = 0.5 * fs
  normal_cutoff = cutoff / nyq
  b, a = cheby1(order, rp, normal_cutoff, btype='low', analog=False)
  return filtfilt(b, a, df[column_name].values).astype("float32")

Kalman Filter

In [12]:
def apply_kalman_filter(df, column_name):
  # Initialize Kalman filter
  kf = KalmanFilter(dim_x=2, dim_z=1)
  kf.x = np.array([df[column_name].iloc[0], 0])  # Initial state (position and velocity)
  kf.F = np.array([[1, 1],
                    [0, 1]])  # State transition matrix
  kf.H = np.array([[1, 0]])  # Measurement matrix
  kf.R = np.array([[3]])  # Measurement noise covariance
  kf.Q = Q_discrete_white_noise(dim=2, dt=1, var=0.5)  # Process noise covariance
  # Initial uncertainty
  kf.P = np.array([[1000., 0.],
                 [0., 1000.]])  # High initial uncertainty

  # Apply Kalman filter
  filtered_data = []
  for value in df[column_name].values:
    kf.predict()
    kf.update(np.array([value]))
    filtered_data.append(kf.x[0])

  return filtered_data.astype("float32")

# Preparing data for trainig


## Loading data


In [13]:
data_path = "/kaggle/input/wi-fi-csi-and-human-activity-for-rooms-abcd"

In [14]:
amplitude_RA = pd.read_csv(f"{data_path}/amplitude_RA.csv").apply(lambda col: col.astype('float32') if col.dtype == 'float64' else col)

In [15]:
amplitude_RB_door_close = pd.read_csv(f"{data_path}/amplitude_RB_door_close.csv").apply(lambda col: col.astype('float32') if col.dtype == 'float64' else col)

amplitude_RB_door_close = amplitude_RB_door_close.drop(
    columns=["25"]
)  # Two non-zero values are masking this column from being dropped

In [16]:
amplitude_RB = pd.read_csv(f"{data_path}/amplitude_RB.csv").apply(lambda col: col.astype('float32') if col.dtype == 'float64' else col)

amplitude_RB = amplitude_RB.drop(
    columns=["25"]
)  # Two non-zero values are masking this column from being dropped

In [17]:
amplitude_RC = pd.read_csv(f"{data_path}/amplitude_RC.csv").apply(lambda col: col.astype('float32') if col.dtype == 'float64' else col)

In [18]:
amplitude_RD = pd.read_csv(f"{data_path}/amplitude_RD.csv").apply(lambda col: col.astype('float32') if col.dtype == 'float64' else col)

## Preprocessing



In [19]:
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.decomposition import PCA

In [20]:
T = 200
use_pca = True
denoising_filter = BUTTER_LOWPASS

In [21]:
def denoise(df, denoising_filter):
    if denoising_filter == BUTTER_LOWPASS:
        for i in range(df.shape[1]):
            df[i] = apply_butter_lowpass_filter(df, i)
    elif denoising_filter == SAVITZKY_GOLAY:
        for i in range(df.shape[1]):
            df[i] = apply_savitzky_golay_filter(df, i)
    elif denoising_filter == GAUSSIAN:
        for i in range(df.shape[1]):
            df[i] = apply_gaussian_filter(df, i)
    elif denoising_filter == WAVELET:
        for i in range(df.shape[1]):
            df[i] = apply_wavelet_denoising(df, i, 'coif5', 9)
    elif denoising_filter == WIENER:
        for i in range(df.shape[1]):
            df[i] = apply_wiener_filter(df, i)
    elif denoising_filter == CHEBYSHEV:
        for i in range(df.shape[1]):
            df[i] = apply_chebyshev_filter(df, i)
    elif denoising_filter == KALMAN:
        for i in range(df.shape[1]):
            df[i] = apply_kalman_filter(df, i)
    elif denoising_filter == BUTTER_CHEBYSHEV:
        for i in range(df.shape[1]):
            df[i] = apply_butter_lowpass_filter(df, i)
            df[i] = apply_chebyshev_filter(df, i)
            
    return df

In [22]:
def reshape(X, y, timestep, overlap_ratio):
    X = np.array(X)
    y = np.array(y)

    label_mapping = {
        "empty": 0,
        "sitting": 1,
        "lying": 2,
        "standing": 3,
        "walking": 4
    }

    y_numeric = np.array([label_mapping[label] for label in y])

    step = int(timestep * (1 - overlap_ratio))

    if step < 1:
        step = 1

    onehot_encoder = OneHotEncoder(sparse_output=False, categories='auto')
    y_onehot = onehot_encoder.fit_transform(y_numeric.reshape(-1, 1))

    X_sequences = []
    y_sequences = []

    for start_idx in range(0, len(X) - timestep + 1, step):
        end_idx = start_idx + timestep
        X_seq = X[start_idx:end_idx]
        y_seq = y_numeric[start_idx:end_idx]

        if np.all(y_seq == y_seq[0]):
            X_sequences.append(X_seq)
            y_sequences.append(y_onehot[start_idx])

    X_final = np.array(X_sequences)
    y_final = np.array(y_sequences)

    return X_final.astype("float32"), y_final.astype("float32")

In [23]:
def preprocess(
    df: pd.DataFrame,
    timestep: int,
    overlap_ratio: int,
    denoising_filter: str,
    use_pca: bool
):
    y = df['human_state']
    X = df.drop('human_state', axis=1)
    X.columns = range(X.shape[1])  # Saving to csv converted the names to chars

    X = denoise(X, denoising_filter)

    X = MinMaxScaler().fit_transform(X)

    if use_pca:
        pca = PCA(n_components=8)
        X = pca.fit_transform(X)

    X, y = reshape(X, y, timestep, overlap_ratio)

    return X.astype("float32"), y.astype("float32")

In [24]:
X_RA_amplitude_denoised_sliding_sequences, y_RA_amplitude_denoised_sliding_sequences = preprocess(amplitude_RA, T, 0.75, denoising_filter, use_pca)
X_RB_door_close_amplitude_denoised_sliding_sequences, y_RB_door_close_amplitude_denoised_sliding_sequences = preprocess(amplitude_RB_door_close, T, 0.75, denoising_filter, use_pca)
X_RB_amplitude_denoised_sliding_sequences, y_RB_amplitude_denoised_sliding_sequences = preprocess(amplitude_RB, T, 0.75, denoising_filter, use_pca)
X_RC_amplitude_denoised_sliding_sequences, y_RC_amplitude_denoised_sliding_sequences = preprocess(amplitude_RC, T, 0.75, denoising_filter, use_pca)
X_RD_amplitude_denoised_sliding_sequences, y_RD_amplitude_denoised_sliding_sequences = preprocess(amplitude_RD, T, 0.75, denoising_filter, use_pca)

X_RA_amplitude_denoised_exclusive_sequences, y_RA_amplitude_denoised_exclusive_sequences = preprocess(amplitude_RA, T, 0, denoising_filter, use_pca)
X_RB_door_close_amplitude_denoised_exclusive_sequences, y_RB_door_close_amplitude_denoised_exclusive_sequences = preprocess(amplitude_RB_door_close, T, 0, denoising_filter, use_pca)
X_RB_amplitude_denoised_exclusive_sequences, y_RB_amplitude_denoised_exclusive_sequences = preprocess(amplitude_RB, T, 0, denoising_filter, use_pca)
X_RC_amplitude_denoised_exclusive_sequences, y_RC_amplitude_denoised_exclusive_sequences = preprocess(amplitude_RC, T, 0, denoising_filter, use_pca)
X_RD_amplitude_denoised_exclusive_sequences, y_RD_amplitude_denoised_exclusive_sequences = preprocess(amplitude_RD, T, 0, denoising_filter, use_pca)

## Create splits


In [25]:
from sklearn.model_selection import train_test_split

In [26]:
seed = 42  # For reproducubility

In [27]:
def train_val_test_split(X, y):
    (
        X_train,
        X_temp,
        y_train,
        y_temp,
    ) = train_test_split(X, y, test_size=0.3, random_state=seed)

    (
        X_val,
        X_test,
        y_val,
        y_test,
    ) = train_test_split(X_temp, y_temp, test_size=0.5, random_state=seed)

    return X_train.astype("float32"), y_train.astype("float32"), X_val.astype("float32"), y_val.astype("float32"), X_test.astype("float32"), y_test.astype("float32")

### Sliding Sequences


In [28]:
(
    X_train_RA_amplitude_denoised_sliding,
    y_train_RA_amplitude_denoised_sliding,
    X_val_RA_amplitude_denoised_sliding,
    y_val_RA_amplitude_denoised_sliding,
    X_test_RA_amplitude_denoised_sliding,
    y_test_RA_amplitude_denoised_sliding,
) = train_val_test_split(
    X_RA_amplitude_denoised_sliding_sequences, y_RA_amplitude_denoised_sliding_sequences
)

In [29]:
del X_RA_amplitude_denoised_sliding_sequences, y_RA_amplitude_denoised_sliding_sequences

In [30]:
(
    X_train_RB_door_close_amplitude_denoised_sliding,
    y_train_RB_door_close_amplitude_denoised_sliding,
    X_val_RB_door_close_amplitude_denoised_sliding,
    y_val_RB_door_close_amplitude_denoised_sliding,
    X_test_RB_door_close_amplitude_denoised_sliding,
    y_test_RB_door_close_amplitude_denoised_sliding,
) = train_val_test_split(
    X_RB_door_close_amplitude_denoised_sliding_sequences,
    y_RB_door_close_amplitude_denoised_sliding_sequences,
)

In [31]:
del X_RB_door_close_amplitude_denoised_sliding_sequences, y_RB_door_close_amplitude_denoised_sliding_sequences

In [32]:
(
    X_train_RB_amplitude_denoised_sliding,
    y_train_RB_amplitude_denoised_sliding,
    X_val_RB_amplitude_denoised_sliding,
    y_val_RB_amplitude_denoised_sliding,
    X_test_RB_amplitude_denoised_sliding,
    y_test_RB_amplitude_denoised_sliding,
) = train_val_test_split(
    X_RB_amplitude_denoised_sliding_sequences, y_RB_amplitude_denoised_sliding_sequences
)

In [33]:
del X_RB_amplitude_denoised_sliding_sequences, y_RB_amplitude_denoised_sliding_sequences

In [34]:
(
    X_train_RC_amplitude_denoised_sliding,
    y_train_RC_amplitude_denoised_sliding,
    X_val_RC_amplitude_denoised_sliding,
    y_val_RC_amplitude_denoised_sliding,
    X_test_RC_amplitude_denoised_sliding,
    y_test_RC_amplitude_denoised_sliding,
) = train_val_test_split(
    X_RC_amplitude_denoised_sliding_sequences, y_RC_amplitude_denoised_sliding_sequences
)

In [35]:
del X_RC_amplitude_denoised_sliding_sequences, y_RC_amplitude_denoised_sliding_sequences

In [36]:
(
    X_train_RD_amplitude_denoised_sliding,
    y_train_RD_amplitude_denoised_sliding,
    X_val_RD_amplitude_denoised_sliding,
    y_val_RD_amplitude_denoised_sliding,
    X_test_RD_amplitude_denoised_sliding,
    y_test_RD_amplitude_denoised_sliding,
) = train_val_test_split(
    X_RD_amplitude_denoised_sliding_sequences, y_RD_amplitude_denoised_sliding_sequences
)

In [37]:
del X_RD_amplitude_denoised_sliding_sequences, y_RD_amplitude_denoised_sliding_sequences

### Exclusive Sequences


In [38]:
(
    X_train_RA_amplitude_denoised_exclusive,
    y_train_RA_amplitude_denoised_exclusive,
    X_val_RA_amplitude_denoised_exclusive,
    y_val_RA_amplitude_denoised_exclusive,
    X_test_RA_amplitude_denoised_exclusive,
    y_test_RA_amplitude_denoised_exclusive,
) = train_val_test_split(
    X_RA_amplitude_denoised_exclusive_sequences, y_RA_amplitude_denoised_exclusive_sequences
)

In [39]:
del X_RA_amplitude_denoised_exclusive_sequences, y_RA_amplitude_denoised_exclusive_sequences

In [40]:
(
    X_train_RB_door_close_amplitude_denoised_exclusive,
    y_train_RB_door_close_amplitude_denoised_exclusive,
    X_val_RB_door_close_amplitude_denoised_exclusive,
    y_val_RB_door_close_amplitude_denoised_exclusive,
    X_test_RB_door_close_amplitude_denoised_exclusive,
    y_test_RB_door_close_amplitude_denoised_exclusive,
) = train_val_test_split(
    X_RB_door_close_amplitude_denoised_exclusive_sequences,
    y_RB_door_close_amplitude_denoised_exclusive_sequences,
)

In [41]:
del X_RB_door_close_amplitude_denoised_exclusive_sequences, y_RB_door_close_amplitude_denoised_exclusive_sequences

In [42]:
(
    X_train_RB_amplitude_denoised_exclusive,
    y_train_RB_amplitude_denoised_exclusive,
    X_val_RB_amplitude_denoised_exclusive,
    y_val_RB_amplitude_denoised_exclusive,
    X_test_RB_amplitude_denoised_exclusive,
    y_test_RB_amplitude_denoised_exclusive,
) = train_val_test_split(
    X_RB_amplitude_denoised_exclusive_sequences, y_RB_amplitude_denoised_exclusive_sequences
)

In [43]:
del X_RB_amplitude_denoised_exclusive_sequences, y_RB_amplitude_denoised_exclusive_sequences

In [44]:
(
    X_train_RC_amplitude_denoised_exclusive,

    y_train_RC_amplitude_denoised_exclusive,
    X_val_RC_amplitude_denoised_exclusive,
    y_val_RC_amplitude_denoised_exclusive,
    X_test_RC_amplitude_denoised_exclusive,
    y_test_RC_amplitude_denoised_exclusive,
) = train_val_test_split(
    X_RC_amplitude_denoised_exclusive_sequences, y_RC_amplitude_denoised_exclusive_sequences
)

In [45]:
del X_RC_amplitude_denoised_exclusive_sequences, y_RC_amplitude_denoised_exclusive_sequences

In [46]:
(
    X_train_RD_amplitude_denoised_exclusive,
    y_train_RD_amplitude_denoised_exclusive,
    X_val_RD_amplitude_denoised_exclusive,
    y_val_RD_amplitude_denoised_exclusive,
    X_test_RD_amplitude_denoised_exclusive,
    y_test_RD_amplitude_denoised_exclusive,
) = train_val_test_split(
    X_RD_amplitude_denoised_exclusive_sequences, y_RD_amplitude_denoised_exclusive_sequences
)

In [47]:
del X_RD_amplitude_denoised_exclusive_sequences, y_RD_amplitude_denoised_exclusive_sequences

# Training Neural Networks



## Utilities


In [ ]:
strategy = tf.distribute.MirroredStrategy()

print(f"Number of devices: {strategy.num_replicas_in_sync}")

In [ ]:
def plot_loss_history(history):
    plt.plot(history.history["loss"], label="train loss")
    plt.plot(history.history["val_loss"], label="val loss")
    plt.legend()
    plt.title("Model Loss")
    plt.show()

    plt.plot(history.history["accuracy"], label="train accuracy")
    plt.plot(history.history["val_accuracy"], label="val accuracy")
    plt.legend()
    plt.title("Model Accuracy")
    plt.show()

In [ ]:
def get_label_of_highest_pred(numbers):
    label_map = {0: "empty", 1: "lying", 2: "sitting", 3: "standing", 4: "walking"}
    highest_index = 0
    for i in range(len(numbers)):
        if numbers[i] > numbers[highest_index]:
            highest_index = i
    return label_map[highest_index]

In [ ]:
def plot_accuracy(y_pred, X_test, y):
    y_test_label = []
    for i in y_test_exclusive_sequences:
        y_test_label.append(get_label_of_highest_pred[i])
    print(y_test_label)
    # accuracies = []
    # for i in range(5):
    #     correct_predictions = np.sum((y_test == i) & (y_pred == i))
    #     total_predictions = np.sum(y_test == i)
    #     accuracy = (
    #         correct_predictions / total_predictions if total_predictions > 0 else 0
    #     )
    #     accuracies.append(accuracy)

    # # Plot the accuracy for each output
    # plt.bar(range(5), accuracies)
    # plt.xlabel("Output")
    # plt.ylabel("Accuracy")
    # plt.title("Accuracy for Each Output")
    # plt.xticks(range(5), [str(i) for i in range(5)])
    # plt.show()

In [ ]:
def set_splits(room_name: str):
    X_train_sliding = eval(f"X_train_R{room_name}_amplitude_denoised_sliding")
    y_train_sliding = eval(f"y_train_R{room_name}_amplitude_denoised_sliding")
    X_train_exclusive = eval(f"X_train_R{room_name}_amplitude_denoised_exclusive")
    y_train_exclusive = eval(f"y_train_R{room_name}_amplitude_denoised_exclusive")

    X_val_sliding = eval(f"X_val_R{room_name}_amplitude_denoised_sliding")
    y_val_sliding = eval(f"y_val_R{room_name}_amplitude_denoised_sliding")
    X_val_exclusive = eval(f"X_val_R{room_name}_amplitude_denoised_exclusive")
    y_val_exclusive = eval(f"y_val_R{room_name}_amplitude_denoised_exclusive")

    X_test_sliding = eval(f"X_test_R{room_name}_amplitude_denoised_sliding")
    y_test_sliding = eval(f"y_test_R{room_name}_amplitude_denoised_sliding")
    X_test_exclusive = eval(f"X_test_R{room_name}_amplitude_denoised_exclusive")
    y_test_exclusive = eval(f"y_test_R{room_name}_amplitude_denoised_exclusive")
    
    return X_train_sliding, y_train_sliding, X_train_exclusive, y_train_exclusive, X_val_sliding, y_val_sliding, X_val_exclusive, y_val_exclusive, X_test_sliding, y_test_sliding, X_test_exclusive, y_test_exclusive

## Set Data Splits

In [ ]:
room_name = "A"

In [ ]:
X_train_sliding, y_train_sliding, X_train_exclusive, y_train_exclusive, X_val_sliding, y_val_sliding, X_val_exclusive, y_val_exclusive, X_test_sliding, y_test_sliding, X_test_exclusive, y_test_exclusive = set_splits(room_name)

## RNN



In [ ]:
from tensorflow.keras import Input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout, LayerNormalization
from tensorflow.keras.initializers import GlorotUniform, Orthogonal, Constant
from tensorflow.keras.regularizers import l2

### Model Architecture


In [ ]:
# Hyperparameters
learning_rate = 0.0001
l2_reg = 0.0001
epochs = 20
batch_size = 32
momentum = 0.9
num_classes = y_train_exclusive.shape[1]
input_features = X_train_exclusive.shape[2]

In [ ]:
def create_rnn():
    inputs = Input(shape=(T, input_features))

    x = SimpleRNN(64, return_sequences=True, kernel_regularizer=l2(l2_reg), kernel_initializer=GlorotUniform(), recurrent_initializer=Orthogonal())(inputs)
    x = LayerNormalization()(x)
    x = Dropout(0.25)(x)

    x = SimpleRNN(128, kernel_regularizer=l2(l2_reg), kernel_initializer=GlorotUniform(), recurrent_initializer=Orthogonal())(x)
    x = LayerNormalization()(x)
    x = Dropout(0.4)(x)

    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=inputs, outputs=outputs)

    optimizer_sgd = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=momentum)
    optimizer_adamw = tf.keras.optimizers.AdamW(learning_rate=learning_rate)
    model.compile(optimizer=optimizer_adamw, loss="categorical_crossentropy", metrics=["accuracy"])

    return model

### On Exclusive Sequences


In [ ]:
# with strategy.scope():
#     model = create_rnn()

model = create_rnn()

In [ ]:
early_stopping_5 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train_exclusive,
    y_train_exclusive,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val_exclusive, y_val_exclusive),
    callbacks=[early_stopping_5],
)

In [ ]:
model.save(
    F"/kaggle/working/RNN_R{room_name}_amplitude_{denoising_filter}_exclusive_sequences.keras"
)

In [ ]:
plot_loss_history(history)

#### Evaluation


In [ ]:
print(f"RNN trained on Exclusive Sequences, tested on Exclusive Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_exclusive"), eval(f"y_test_RA_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_exclusive"), eval(f"y_test_RB_door_close_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_exclusive"), eval(f"y_test_RB_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_exclusive"), eval(f"y_test_RC_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_exclusive"), eval(f"y_test_RD_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")



print(f"RNN trained on Exclusive Sequences, tested on Sliding Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_sliding"), eval(f"y_test_RA_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_sliding"), eval(f"y_test_RB_door_close_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_sliding"), eval(f"y_test_RB_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_sliding"), eval(f"y_test_RC_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_sliding"), eval(f"y_test_RD_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")

### On Sliding sequences

In [ ]:
model = create_rnn()

In [ ]:
early_stopping_5 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train_sliding,
    y_train_sliding,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(
        X_val_sliding,
        y_val_sliding,
    ),
    callbacks=[early_stopping_5],
)

In [ ]:
model.save(
    f"/kaggle/working/RNN_R{room_name}_amplitude_{denoising_filter}_sliding_sequences.keras"
)

In [ ]:
plot_loss_history(history)

#### Evaluation


In [ ]:
print(f"RNN trained on Sliding Sequences, tested on Exclusive Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_exclusive"), eval(f"y_test_RA_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_exclusive"), eval(f"y_test_RB_door_close_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_exclusive"), eval(f"y_test_RB_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_exclusive"), eval(f"y_test_RC_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_exclusive"), eval(f"y_test_RD_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")



print(f"RNN trained on Sliding Sequences, tested on Sliding Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_sliding"), eval(f"y_test_RA_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_sliding"), eval(f"y_test_RB_door_close_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_sliding"), eval(f"y_test_RB_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_sliding"), eval(f"y_test_RC_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_sliding"), eval(f"y_test_RD_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")

## GRU


In [ ]:
from tensorflow.keras import Input, regularizers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GRU, LayerNormalization
from tensorflow.keras.initializers import GlorotUniform, Orthogonal, Constant

### Model architecture

In [ ]:
# Hyperparameters
learning_rate = 0.0001
l2_reg = 0.0001
epochs = 20
batch_size = 32
momentum = 0.9
num_classes = y_train_exclusive.shape[1]
input_features = X_train_exclusive.shape[2]

In [ ]:
def create_gru():
    inputs = Input(shape=(T, input_features))

    x = GRU(64, return_sequences=True, kernel_regularizer=l2(l2_reg), kernel_initializer=GlorotUniform(), recurrent_initializer=Orthogonal())(inputs)
    x = LayerNormalization()(x)
    x = Dropout(0.25)(x)

    x = GRU(128, kernel_regularizer=l2(l2_reg), kernel_initializer=GlorotUniform(), recurrent_initializer=Orthogonal())(x)
    x = LayerNormalization()(x)
    x = Dropout(0.4)(x)

    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=inputs, outputs=outputs)

    optimizer_sgd = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=momentum)
    optimizer_adamw = tf.keras.optimizers.AdamW(learning_rate=learning_rate)
    model.compile(optimizer=optimizer_adamw, loss="categorical_crossentropy", metrics=["accuracy"])

    return model

### On Exclusive Sequences


In [ ]:
model = create_gru()

In [ ]:
early_stopping_5 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train_exclusive,
    y_train_exclusive,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val_exclusive, y_val_exclusive),
    callbacks=[early_stopping_5],
)

In [ ]:
model.save(
    f"/kaggle/working/GRU_R{room_name}_amplitude_{denoising_filter}_exclusive_sequences.keras"
)

In [ ]:
plot_loss_history(history)

#### Evaluation


In [ ]:
print(f"GRU trained on Exclusive Sequences, tested on Exclusive Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_exclusive"), eval(f"y_test_RA_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_exclusive"), eval(f"y_test_RB_door_close_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_exclusive"), eval(f"y_test_RB_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_exclusive"), eval(f"y_test_RC_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_exclusive"), eval(f"y_test_RD_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")



print(f"GRU trained on Exclusive Sequences, tested on Sliding Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_sliding"), eval(f"y_test_RA_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_sliding"), eval(f"y_test_RB_door_close_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_sliding"), eval(f"y_test_RB_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_sliding"), eval(f"y_test_RC_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_sliding"), eval(f"y_test_RD_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")

### On Sliding Sequences


In [ ]:
model = create_gru()

In [ ]:
early_stopping_5 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train_sliding,
    y_train_sliding,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val_sliding, y_val_sliding),
    callbacks=[early_stopping_5],
)

In [ ]:
model.save(
    f"/kaggle/working/GRU_R{room_name}_amplitude_{denoising_filter}_sliding_sequences.keras"
)

In [ ]:
plot_loss_history(history)

#### Evaluation


In [ ]:
print(f"GRU trained on Sliding Sequences, tested on Exclusive Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_exclusive"), eval(f"y_test_RA_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_exclusive"), eval(f"y_test_RB_door_close_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_exclusive"), eval(f"y_test_RB_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_exclusive"), eval(f"y_test_RC_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_exclusive"), eval(f"y_test_RD_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")



print(f"GRU trained on Sliding Sequences, tested on Sliding Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_sliding"), eval(f"y_test_RA_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_sliding"), eval(f"y_test_RB_door_close_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_sliding"), eval(f"y_test_RB_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_sliding"), eval(f"y_test_RC_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_sliding"), eval(f"y_test_RD_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")

## LSTM



In [ ]:
from tensorflow.keras import Input, regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, BatchNormalization

### Model Architecture


In [48]:
# Hyperparameters
learning_rate = 0.0001
l2_reg = 0.0001
epochs = 20
batch_size = 32
momentum = 0.9
num_classes = y_train_exclusive.shape[1]
input_features = X_train_exclusive.shape[2]

NameError: name 'y_train_exclusive' is not defined

In [ ]:
def create_lstm():
    inputs = Input(shape=(T, input_features))

    x = LSTM(64, return_sequences=True, 
             kernel_regularizer=l2(l2_reg),
             kernel_initializer=GlorotUniform(),
             recurrent_initializer=Orthogonal(),)(inputs)
    x = LayerNormalization()(x)
    x = Dropout(0.25)(x)

    x = LSTM(128, kernel_regularizer=l2(l2_reg), 
             kernel_initializer=GlorotUniform(), 
             recurrent_initializer=Orthogonal(),)(x)
    x = LayerNormalization()(x)
    x = Dropout(0.4)(x)

    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=inputs, outputs=outputs)

    optimizer_sgd = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=momentum)
    optimizer_adamw = tf.keras.optimizers.AdamW(learning_rate=learning_rate)
    model.compile(optimizer=optimizer_adamw, loss="categorical_crossentropy", metrics=["accuracy"])

    return model

### On Exclusive Sequences


In [ ]:
model = create_lstm()

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train_exclusive,
    y_train_exclusive,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val_exclusive, y_val_exclusive),
    callbacks=[early_stopping],
)

In [ ]:
model.save(
    f"/kaggle/working/LSTM_R{room_name}_amplitude_{denoising_filter}_exclusive_sequences.keras"
)

In [ ]:
plot_loss_history(history)

#### Evaluation

In [ ]:
print(f"LSTM trained on Exclusive Sequences, tested on Exclusive Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_exclusive"), eval(f"y_test_RA_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_exclusive"), eval(f"y_test_RB_door_close_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_exclusive"), eval(f"y_test_RB_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_exclusive"), eval(f"y_test_RC_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_exclusive"), eval(f"y_test_RD_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")



print(f"LSTM trained on Exclusive Sequences, tested on Sliding Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_sliding"), eval(f"y_test_RA_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_sliding"), eval(f"y_test_RB_door_close_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_sliding"), eval(f"y_test_RB_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_sliding"), eval(f"y_test_RC_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_sliding"), eval(f"y_test_RD_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")

### On Sliding Sequences


In [ ]:
model = create_lstm()

In [ ]:
early_stopping_5 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train_sliding,
    y_train_sliding,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val_sliding, y_val_sliding),
    callbacks=[early_stopping_5],
)

In [ ]:
model.save(
    f"/kaggle/working/LSTM_R{room_name}_amplitude_{denoising_filter}_sliding_sequences.keras"
)

In [ ]:
plot_loss_history(history)

#### Evaluation

In [ ]:
print(f"LSTM trained on Sliding Sequences, tested on Exclusive Sequences - {denoising_filter}")
loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_exclusive"), eval(f"y_test_RA_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_exclusive"), eval(f"y_test_RB_door_close_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_exclusive"), eval(f"y_test_RB_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_exclusive"), eval(f"y_test_RC_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_exclusive"), eval(f"y_test_RD_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")



print(f"LSTM trained on Sliding Sequences, tested on Sliding Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_sliding"), eval(f"y_test_RA_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_sliding"), eval(f"y_test_RB_door_close_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_sliding"), eval(f"y_test_RB_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_sliding"), eval(f"y_test_RC_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_sliding"), eval(f"y_test_RD_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")

## BLSTM


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    LSTM,
    Bidirectional,
    BatchNormalization,
    Input,
)
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.regularizers import l2

### Model Architecture


In [ ]:
# Hyperparameters
learning_rate = 0.0001
l2_reg = 0.0001
epochs = 20
batch_size = 32
momentum = 0.9
num_classes = y_train_exclusive.shape[1]
input_features = X_train_exclusive.shape[2]

In [ ]:
def create_blstm():
    inputs = Input(shape=(T, input_features))

    x = Bidirectional(
            LSTM(
             64,
             return_sequences=True, 
             kernel_regularizer=l2(l2_reg),
             kernel_initializer=GlorotUniform(),
             recurrent_initializer=Orthogonal()
            ))(inputs)
    x = LayerNormalization()(x)
    x = Dropout(0.25)(x)

    x = Bidirectional(
            LSTM(
             128,
             kernel_regularizer=l2(l2_reg), 
             kernel_initializer=GlorotUniform(), 
             recurrent_initializer=Orthogonal()
            ))(x)
    x = LayerNormalization()(x)
    x = Dropout(0.4)(x)

    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=inputs, outputs=outputs)

    optimizer_sgd = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=momentum)
    optimizer_adamw = tf.keras.optimizers.AdamW(learning_rate=learning_rate)
    model.compile(optimizer=optimizer_adamw, loss="categorical_crossentropy", metrics=["accuracy"])

    return model

### On Exclusive Sequences


In [ ]:
model = create_blstm()

In [ ]:
early_stopping_5 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train_exclusive,
    y_train_exclusive,
    validation_data=(X_val_exclusive, y_val_exclusive),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping_5],
)

In [ ]:
model.save(
    f"/kaggle/working/BLSTM_R{room_name}_amplitude_{denoising_filter}_exclsuive_sequences.keras"
)

In [ ]:
plot_loss_history(history)

#### Evaluation

In [ ]:
print(f"BLSTM trained on Exclusive Sequences, tested on Exclusive Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_exclusive"), eval(f"y_test_RA_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_exclusive"), eval(f"y_test_RB_door_close_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_exclusive"), eval(f"y_test_RB_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_exclusive"), eval(f"y_test_RC_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_exclusive"), eval(f"y_test_RD_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")



print(f"BLSTM trained on Exclusive Sequences, tested on Sliding Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_sliding"), eval(f"y_test_RA_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_sliding"), eval(f"y_test_RB_door_close_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_sliding"), eval(f"y_test_RB_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_sliding"), eval(f"y_test_RC_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_sliding"), eval(f"y_test_RD_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")

### On Sliding Sequences


In [ ]:
model = create_blstm()

In [ ]:
early_stopping_5 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train_sliding,
    y_train_sliding,
    validation_data=(X_val_sliding, y_val_sliding),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping_5],
)

In [ ]:
model.save(
    f"/kaggle/working/BLSTM_R{room_name}_amplitude_{denoising_filter}_sliding_sequences.keras"
)

In [ ]:
plot_loss_history(history)

#### Evaluation

In [ ]:
print(f"BLSTM trained on Sliding Sequences, tested on Exclusive Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_exclusive"), eval(f"y_test_RA_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_exclusive"), eval(f"y_test_RB_door_close_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_exclusive"), eval(f"y_test_RB_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_exclusive"), eval(f"y_test_RC_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_exclusive"), eval(f"y_test_RD_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")



print(f"BLSTM trained on Sliding Sequences, tested on Sliding Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_sliding"), eval(f"y_test_RA_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_sliding"), eval(f"y_test_RB_door_close_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_sliding"), eval(f"y_test_RB_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_sliding"), eval(f"y_test_RC_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_sliding"), eval(f"y_test_RD_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")

## CNN + BLSTM



In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    LSTM,
    Dense,
    Dropout,
    MaxPooling1D,
    Bidirectional,
    BatchNormalization,
    Add,
)

### Model Architecture


In [ ]:
# Hyperparameters
learning_rate = 0.0001
l2_reg = 0.0001
epochs = 20
batch_size = 32
momentum = 0.9
num_classes = y_train_exclusive.shape[1]
input_features = X_train_exclusive.shape[2]

In [ ]:
def create_cnnblstm():
    inputs = Input(shape=(T, input_features))

    x = Conv1D(filters=64, kernel_size=3, activation="relu")(inputs)
    x = MaxPooling1D(pool_size=2)(x)
    x = Conv1D(filters=64, kernel_size=3, activation="relu")(x)
    x = MaxPooling1D(pool_size=2)(x)

    x = Bidirectional(
          LSTM(64,
             return_sequences=True, 
             kernel_regularizer=l2(l2_reg),
             kernel_initializer=GlorotUniform(),
             recurrent_initializer=Orthogonal(),
          ))(x)
    x = LayerNormalization()(x)
    x = Dropout(0.25)(x)

    x = Bidirectional(
          LSTM(128,
             return_sequences=False,
             kernel_regularizer=l2(l2_reg), 
             kernel_initializer=GlorotUniform(), 
             recurrent_initializer=Orthogonal(),
          ))(x)
    x = LayerNormalization()(x)
    x = Dropout(0.4)(x)
    
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=inputs, outputs=outputs)

    optimizer_adamw = tf.keras.optimizers.AdamW(learning_rate=learning_rate)
    model.compile(optimizer=optimizer_adamw, loss="categorical_crossentropy", metrics=["accuracy"])

    return model

### On Exclusive Sequences


In [ ]:
model = create_cnnblstm()

In [ ]:
early_stopping_5 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train_exclusive,
    y_train_exclusive,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val_exclusive, y_val_exclusive),
    callbacks=[early_stopping_5],
)

In [ ]:
model.save(
    f"/kaggle/working/CNN_BLSTM_R{room_name}_amplitude_{denoising_filter}_exclusive_sequences.keras"
)

In [ ]:
plot_loss_history(history)

#### Evaluation

In [ ]:
print(f"CNN-BLSTM trained on Exclusive Sequences, tested on Exclusive Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_exclusive"), eval(f"y_test_RA_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_exclusive"), eval(f"y_test_RB_door_close_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_exclusive"), eval(f"y_test_RB_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_exclusive"), eval(f"y_test_RC_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_exclusive"), eval(f"y_test_RD_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")



print(f"CNN-BLSTM trained on Exclusive Sequences, tested on Sliding Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_sliding"), eval(f"y_test_RA_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_sliding"), eval(f"y_test_RB_door_close_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_sliding"), eval(f"y_test_RB_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_sliding"), eval(f"y_test_RC_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_sliding"), eval(f"y_test_RD_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")

### On Sliding Sequences


In [ ]:
model = create_cnnblstm()

In [ ]:
early_stopping_5 = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train_sliding,
    y_train_sliding,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val_sliding, y_val_sliding),
    callbacks=[early_stopping_5],
)

In [ ]:
model.save(
    f"/kaggle/working/CNN_BLSTM_R{room_name}_amplitude_{denoising_filter}_sliding_sequences.keras"
)

In [ ]:
plot_loss_history(history)

#### Evaluation

In [ ]:
print(f"CNN-BLSTM trained on Sliding Sequences, tested on Exclusive Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_exclusive"), eval(f"y_test_RA_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_exclusive"), eval(f"y_test_RB_door_close_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_exclusive"), eval(f"y_test_RB_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_exclusive"), eval(f"y_test_RC_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_exclusive"), eval(f"y_test_RD_amplitude_denoised_exclusive")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")



print(f"CNN-BLSTM trained on Sliding Sequences, tested on Sliding Sequences - {denoising_filter}")

loss, accuracy = model.evaluate(
    eval(f"X_test_RA_amplitude_denoised_sliding"), eval(f"y_test_RA_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room A: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_door_close_amplitude_denoised_sliding"), eval(f"y_test_RB_door_close_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B (door closed): {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RB_amplitude_denoised_sliding"), eval(f"y_test_RB_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room B: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RC_amplitude_denoised_sliding"), eval(f"y_test_RC_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room C: {(accuracy * 100):.2f}%")

loss, accuracy = model.evaluate(
    eval(f"X_test_RD_amplitude_denoised_sliding"), eval(f"y_test_RD_amplitude_denoised_sliding")
)
print(f"Test Accuracy in room D: {(accuracy * 100):.2f}%")

In [ ]:
# import os
# import zipfile

# def zip_directory(directory_path):
#     # Define the name of the zip file (you can customize the name as needed)
#     zip_file_name = os.path.basename(directory_path.rstrip('/')) + ".zip"
#     zip_file_path = os.path.join(directory_path, zip_file_name)
    
#     # Create a zip file
#     with zipfile.ZipFile(zip_file_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
#         # Traverse through all files in the directory
#         for foldername, subfolders, filenames in os.walk(directory_path):
#             for filename in filenames:
#                 file_path = os.path.join(foldername, filename)
#                 # Skip adding the zip file to itself
#                 if file_path == zip_file_path:
#                     continue
#                 # Add file to the zip archive
#                 zipf.write(file_path, os.path.relpath(file_path, directory_path))
    
#     print(f"Zip file created: {zip_file_path}")

# # Example usage
# directory_to_zip = "/kaggle/working/"  # Replace with your directory path
# zip_directory(directory_to_zip)